<a href="https://colab.research.google.com/github/andinaufal120/kalimantan-frp-prediction/blob/main/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 6. Setup

### 6.1 Imports and Constants

All required libraries are imported here. Constants are defined once and referenced
throughout to ensure reproducibility and consistency across sections.

- `SEED = 42` — random state for RF, XGBoost, and RandomizedSearchCV
- `TRAIN_MONTHS = 12`, `TEST_MONTHS = 3`, `STEP_MONTHS = 2` — sliding window parameters
- `N_ITER = 20` — RandomizedSearchCV iterations per fold

In [ ]:
from typing import Final
from dateutil.relativedelta import relativedelta

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import randint, uniform
from xgboost import XGBRegressor

%matplotlib inline

In [ ]:
SEED: Final[int] = 42
PATH: Final[str] = "./Dataset_ML.csv"
FIG_DIR: Final[str] = "./artifacts/figures/"

TRAIN_MONTHS: Final[int] = 12
TEST_MONTHS: Final[int] = 3
STEP_MONTHS: Final[int] = 2
N_ITER: Final[int] = 20

In [ ]:
df = pd.read_csv(PATH)
df['acq_date'] = pd.to_datetime(df['acq_date'])

row, col = df.shape
print(f"Loaded {row} rows, {col} cols.")

Loaded 59155 rows, 46 cols.


### 6.2 Feature Matrix and Target Vector

The final 14 features are selected from EDA Phase 4 (Spearman correlation analysis).
Target variable is `frp_log1p` (log1p-transformed FRP in MW).
Both `X` and `y` extracted as NumPy arrays for compatibility with sklearn and XGBoost.

In [ ]:
FEATURES: Final[list] = [
    'evi', 'ndvi_age_days', 'lst_day', 'lst_night',
    'land_cover_type', 'peat_type_class', 'ssrd', 'tp_hourly',
    'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos',
    'vpd_kpa', 'wind_speed_m_s'
]

X = df[FEATURES].to_numpy()
y = df['frp_log1p'].to_numpy()

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

X shape: (59155, 14)
y shape: (59155,)


### 6.3 Sliding Window CV Splitter

Custom date-range-based splitter using `dateutil.relativedelta`.
Produces 11 folds with parameters `(train=12, test=3, step=2)` months.
Row-count-based splitting avoided. Fire detections are seasonally uneven,
so fixed row counts do not correspond to fixed time periods.

In [ ]:
def sliding_window_splits(df, date_col, train_months, test_months, step_months):
    dates = df[date_col]
    start = dates.min()
    end = dates.max()

    fold_start = start
    while True:
        train_end = fold_start + relativedelta(months=train_months)
        test_end = train_end + relativedelta(months=test_months)

        if test_end > end:
            break

        train_idx = df.index[(dates >= fold_start) & (dates < train_end)].to_numpy()
        test_idx = df.index[(dates >= train_end) & (dates < test_end)].to_numpy()

        if len(train_idx) > 0 and len(test_idx) > 0:
            yield train_idx, test_idx

        fold_start += relativedelta(months=step_months)

splits = list(sliding_window_splits(
    df, 'acq_date', TRAIN_MONTHS, TEST_MONTHS, STEP_MONTHS
))

print(f"Total folds: {len(splits)}")

Total folds: 11


### 6.4 Evaluation Metrics

Four metrics reported per fold: `rmse_log1p`, `mae_log1p`, `r2`, and `mae_mw`.
`mae_mw` back-transforms predictions via `np.expm1` to report error in original MW units,
providing an operationally interpretable measure alongside log-space metrics.

In [ ]:
def metric_function(y_true, y_pred):
    rmse_log1p = root_mean_squared_error(y_true, y_pred)
    mae_log1p = mean_absolute_error(y_true, y_pred)
    mae_raw = mean_absolute_error(np.expm1(y_true), np.expm1(y_pred))
    r_square = r2_score(y_true, y_pred)

    return rmse_log1p, mae_log1p, mae_raw, r_square

## 7. Random Forest

### 7.1 Hyperparameter Search Space

We use `RandomizedSearchCV` with 20 iterations inside each CV fold,
fitting only on training data to prevent leakage. Search space defined
based on dataset size (59,155 rows, 14 features) and CPU-only constraints.

In [ ]:
rf_param_dist = {
    'n_estimators': randint(100, 300),
    'max_depth': randint(5, 20),
    'max_features': [3, 6, 9, 12],
    'min_samples_leaf': randint(1, 50),
}

### 7.2 Training and Per-Fold Evaluation

Best estimator from `RandomizedSearchCV` refit on the full training window
before predicting on the test window. Metrics collected per fold into
`rf_results` for downstream aggregation.

In [ ]:
rf_results = []

for i, (train_idx, test_idx) in enumerate(splits):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    rf = RandomForestRegressor(random_state=SEED, n_jobs=-1)
    search = RandomizedSearchCV(
        rf,
        param_distributions=rf_param_dist,
        n_iter=N_ITER,
        scoring='neg_root_mean_squared_error',
        cv=3,
        random_state=SEED,
        n_jobs=-1
    )
    search.fit(X_train, y_train)

    y_pred = search.best_estimator_.predict(X_test)
    rmse, mae, mae_raw, r2 = metric_function(y_test, y_pred)

    rf_results.append({
        'fold': i + 1,
        'best_params': search.best_params_,
        'rmse_log1p': rmse,
        'mae_log1p': mae,
        'mae_mw': mae_raw,
        'r2': r2,
        'train_size': len(train_idx),
        'test_size': len(test_idx),
    })

    print(f"Fold {i+1:2d} — RMSE: {rmse:.4f}, MAE: {mae:.4f}, R²: {r2:.4f}, MAE(MW): {mae_raw:.2f}")

Fold  1 — RMSE: 0.5198, MAE: 0.3999, R²: 0.4769, MAE(MW): 2.92
Fold  2 — RMSE: 0.4915, MAE: 0.3789, R²: 0.4713, MAE(MW): 2.55
Fold  3 — RMSE: 0.6273, MAE: 0.4634, R²: 0.3692, MAE(MW): 4.70
Fold  4 — RMSE: 0.6358, MAE: 0.4681, R²: 0.3248, MAE(MW): 4.68
Fold  5 — RMSE: 0.6042, MAE: 0.4388, R²: 0.3587, MAE(MW): 3.84
Fold  6 — RMSE: 0.5642, MAE: 0.4325, R²: 0.4642, MAE(MW): 3.44
Fold  7 — RMSE: 0.4538, MAE: 0.3463, R²: 0.5283, MAE(MW): 2.07
Fold  8 — RMSE: 0.4128, MAE: 0.3166, R²: 0.5469, MAE(MW): 1.63
Fold  9 — RMSE: 0.5225, MAE: 0.3914, R²: 0.4679, MAE(MW): 2.87
Fold 10 — RMSE: 0.6919, MAE: 0.5291, R²: 0.2372, MAE(MW): 6.37
Fold 11 — RMSE: 0.6862, MAE: 0.5202, R²: 0.2485, MAE(MW): 6.13


### 7.3 Results Summary

Per-fold and aggregate (mean ± std) metrics printed across all 11 folds.
Per-fold reporting prioritized given severe size imbalance: Folds 4–5
(Aug–Oct 2023, El Niño peak) contain ~47% of all rows.

In [ ]:
rf_df = pd.DataFrame(rf_results)

print("\nPer-fold results:")
print(rf_df[['fold', 'rmse_log1p', 'mae_log1p', 'mae_mw', 'r2', 'train_size', 'test_size']].to_string(index=False))

print("\nAggregate (mean ± std):")
for metric in ['rmse_log1p', 'mae_log1p', 'mae_mw', 'r2']:
    mean = rf_df[metric].mean()
    std = rf_df[metric].std()
    print(f"  {metric:15s}: {mean:.4f} ± {std:.4f}")


Per-fold results:
 fold  rmse_log1p  mae_log1p   mae_mw       r2  train_size  test_size
    1    0.519847   0.399920 2.915033 0.476922        4485        399
    2    0.491503   0.378868 2.554953 0.471286        4572        918
    3    0.627300   0.463395 4.701663 0.369157        4564       3405
    4    0.635844   0.468132 4.684657 0.324843        5388      27894
    5    0.604219   0.438819 3.839998 0.358719       15872      29063
    6    0.564160   0.432546 3.437193 0.464238       42585       1268
    7    0.453806   0.346310 2.073917 0.528298       43391        607
    8    0.412845   0.316561 1.632886 0.546927       43452        647
    9    0.522548   0.391420 2.867350 0.467883       43540        795
   10    0.691910   0.529114 6.373766 0.237233       42684       8101
   11    0.686208   0.520220 6.134575 0.248530       32323       8160

Aggregate (mean ± std):
  rmse_log1p     : 0.5646 ± 0.0927
  mae_log1p      : 0.4259 ± 0.0673
  mae_mw         : 3.7469 ± 1.5701
  r2       

## 8. XGBoost

### 8.1 Hyperparameter Search Space

We use `RandomizedSearchCV` with 20 iterations, consistent with the RF setup.
Search space covers `n_estimators`, `learning_rate`, `max_depth`,
`reg_alpha`, and `reg_lambda` — regularization parameters included
given the dataset's skewed fold size distribution.

In [ ]:
xgboost_param_dist = {
    'n_estimators': randint(100, 500),
    'learning_rate': uniform(0.01, 0.29),
    'max_depth': randint(5, 20),
    'reg_alpha': uniform(0, 10),
    'reg_lambda': uniform(0, 10)
}

### 8.2 Training and Per-Fold Evaluation

Same sliding window CV procedure as Section 7. Results collected
per fold into `xgb_results` for comparison against RF in Section 10.

In [ ]:
xgboost_results = []

for i, (train_idx, test_idx) in enumerate(splits):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    xgboost = XGBRegressor(random_state=SEED, n_jobs=-1)
    search = RandomizedSearchCV(
        xgboost,
        param_distributions=xgboost_param_dist,
        n_iter=N_ITER,
        scoring='neg_root_mean_squared_error',
        cv=3,
        random_state=SEED,
        n_jobs=-1
    )
    search.fit(X_train, y_train)

    y_pred = search.best_estimator_.predict(X_test)
    rmse, mae, mae_raw, r2 = metric_function(y_test, y_pred)

    xgboost_results.append({
        'fold': i + 1,
        'best_params': search.best_params_,
        'rmse_log1p': rmse,
        'mae_log1p': mae,
        'mae_mw': mae_raw,
        'r2': r2,
        'train_size': len(train_idx),
        'test_size': len(test_idx),
    })

    print(f"Fold {i+1:2d} — RMSE: {rmse:.4f}, MAE: {mae:.4f}, R²: {r2:.4f}, MAE(MW): {mae_raw:.2f}")

Fold  1 — RMSE: 0.5566, MAE: 0.4192, R²: 0.4004, MAE(MW): 2.99
Fold  2 — RMSE: 0.5030, MAE: 0.3862, R²: 0.4463, MAE(MW): 2.62
Fold  3 — RMSE: 0.6354, MAE: 0.4723, R²: 0.3529, MAE(MW): 4.78
Fold  4 — RMSE: 0.6343, MAE: 0.4776, R²: 0.3281, MAE(MW): 4.75
Fold  5 — RMSE: 0.5941, MAE: 0.4376, R²: 0.3801, MAE(MW): 3.82
Fold  6 — RMSE: 0.6312, MAE: 0.4844, R²: 0.3293, MAE(MW): 3.75
Fold  7 — RMSE: 0.4893, MAE: 0.3606, R²: 0.4517, MAE(MW): 2.14
Fold  8 — RMSE: 0.4405, MAE: 0.3276, R²: 0.4841, MAE(MW): 1.70
Fold  9 — RMSE: 0.5264, MAE: 0.3928, R²: 0.4601, MAE(MW): 2.86
Fold 10 — RMSE: 0.7009, MAE: 0.5323, R²: 0.2173, MAE(MW): 6.41
Fold 11 — RMSE: 0.7182, MAE: 0.5399, R²: 0.1769, MAE(MW): 6.30


### 8.3 Results Summary

Per-fold and aggregate (mean ± std) metrics across all 11 folds.
Notable: model degrades on Folds 10–11 (2024 dry season) — trained
on El Niño-driven behavior but 2024 extreme events occurred under
neutral ENSO conditions (ONI = −0.17, DMI = 0.05).

In [ ]:
xgboost_df = pd.DataFrame(xgboost_results)

print("\nPer-fold results:")
print(xgboost_df[['fold', 'rmse_log1p', 'mae_log1p', 'mae_mw', 'r2', 'train_size', 'test_size']].to_string(index=False))

print("\nAggregate (mean ± std):")
for metric in ['rmse_log1p', 'mae_log1p', 'mae_mw', 'r2']:
    mean = xgboost_df[metric].mean()
    std = xgboost_df[metric].std()
    print(f"  {metric:15s}: {mean:.4f} ± {std:.4f}")


Per-fold results:
 fold  rmse_log1p  mae_log1p   mae_mw       r2  train_size  test_size
    1    0.556596   0.419207 2.989288 0.400353        4485        399
    2    0.502984   0.386157 2.616410 0.446298        4572        918
    3    0.635355   0.472292 4.778721 0.352850        4564       3405
    4    0.634294   0.477582 4.746431 0.328129        5388      27894
    5    0.594076   0.437601 3.822695 0.380067       15872      29063
    6    0.631230   0.484437 3.748383 0.329279       42585       1268
    7    0.489286   0.360598 2.144147 0.451659       43391        607
    8    0.440536   0.327589 1.695502 0.484109       43452        647
    9    0.526365   0.392773 2.862603 0.460080       43540        795
   10    0.700887   0.532285 6.410338 0.217312       42684       8101
   11    0.718177   0.539865 6.295415 0.176881       32323       8160

Aggregate (mean ± std):
  rmse_log1p     : 0.5845 ± 0.0891
  mae_log1p      : 0.4391 ± 0.0689
  mae_mw         : 3.8282 ± 1.5810
  r2       